# 🚲 Bike-Sharing Demand Analysis & Prediction
## Complete Data Science Mini Project — Google Colab

**Goal:** Analyze historical bike rentals, discover patterns, train regression models, and predict hourly demand.

**Dataset:** UCI Bike Sharing Dataset (`hour.csv`)  
**Target:** `cnt` — total bike rentals in an hour.

### Workflow
1. Problem definition
2. Dataset collection and understanding
3. Data quality inspection
4. Cleaning and preprocessing
5. Exploratory Data Analysis (EDA)
6. Feature preparation
7. Train/test split
8. Model training
9. Model evaluation
10. Findings and conclusion

Run cells from top to bottom. Use actual outputs from your own run in your report.

## 1. Problem definition

Bike-sharing companies need to estimate how many bicycles customers may rent at different hours. Demand can vary with time, season, weather, and working-day patterns.

**Problem type:** Supervised Machine Learning — Regression, because the target is a numerical count.

**Objectives:** understand and clean the dataset, visualize demand patterns, train Linear Regression and Random Forest, compare MAE/RMSE/R², and interpret the results.

## 2. Import libraries

- Pandas and NumPy: data processing
- Matplotlib and Seaborn: visualization
- scikit-learn: preprocessing, model training, evaluation
- Requests and ZipFile: dataset download

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 3. Collect the dataset

This downloads the official UCI Bike Sharing Dataset ZIP and reads the hourly file, `hour.csv`. If download fails, use the manual upload alternative below.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    hourly_file = next(n for n in archive.namelist() if n.endswith("hour.csv"))
    with archive.open(hourly_file) as f:
        raw_df = pd.read_csv(f)

print("Dataset downloaded.")
print("Rows and columns:", raw_df.shape)
display(raw_df.head())

### Alternative: upload `hour.csv` manually if download fails

Run this cell only if needed.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# raw_df = pd.read_csv(next(iter(uploaded)))
# print(raw_df.shape)
# display(raw_df.head())

## 4. Understand the data

Important variables:
- `dteday`: date
- `season`, `mnth`, `hr`, `weekday`: time variables
- `holiday`, `workingday`: calendar indicators
- `weathersit`: weather category
- `temp`, `atemp`, `hum`, `windspeed`: normalized weather measurements
- `casual`, `registered`: types of riders
- `cnt`: total rentals and the target variable

**Important:** `casual + registered = cnt`. Do not use `casual` or `registered` as model inputs, because that would leak the target answer into the model.

In [ ]:
print("Shape:", raw_df.shape)
print("\nColumns:", raw_df.columns.tolist())
print("\nData types and non-null counts:")
raw_df.info()
print("\nMissing values:")
display(raw_df.isna().sum().to_frame("missing_values"))
print("\nDuplicate rows:", raw_df.duplicated().sum())
print("\nSummary statistics:")
display(raw_df.describe().T)

## 5. Clean the data

We standardize column names, parse dates, remove duplicate rows, remove rows with a missing target, keep the target non-negative, and sort chronologically before splitting the data.

In [ ]:
df = raw_df.copy()
df.columns = [str(c).strip().lower() for c in df.columns]
df["dteday"] = pd.to_datetime(df["dteday"], errors="coerce")

before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = before - len(df)
before_target = len(df)
df = df.dropna(subset=["cnt"]).copy()
missing_target_removed = before_target - len(df)
df["cnt"] = df["cnt"].clip(lower=0)
df = df.sort_values(["dteday", "hr"]).reset_index(drop=True)

print("Original rows:", len(raw_df))
print("Duplicates removed:", duplicates_removed)
print("Rows removed for missing target:", missing_target_removed)
print("Rows after basic cleaning:", len(df))
display(df.isna().sum().to_frame("remaining_missing_values"))

## 6. Exploratory Data Analysis (EDA)\nEDA means exploring data with statistics and graphs before modeling.

### 6.1 Distribution of hourly rentals

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df["cnt"], bins=40, kde=True)
plt.title("Distribution of Hourly Bike Rentals")
plt.xlabel("Total rentals in an hour (cnt)")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()
print("Mean:", round(df["cnt"].mean(), 2))
print("Median:", round(df["cnt"].median(), 2))
print("Maximum:", int(df["cnt"].max()))

### 6.2 Average demand by hour

In [ ]:
hourly = df.groupby("hr", as_index=False)["cnt"].mean()
plt.figure(figsize=(10,5))
sns.lineplot(data=hourly, x="hr", y="cnt", marker="o")
plt.title("Average Bike Rentals by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Average rentals")
plt.xticks(range(24))
plt.tight_layout()
plt.show()
peak_hour = int(hourly.loc[hourly["cnt"].idxmax(), "hr"])
print("Highest average demand hour:", f"{peak_hour}:00")

### 6.3 Working days versus non-working days

In [ ]:
df["day_type"] = df["workingday"].map({0:"Non-working day", 1:"Working day"})
plt.figure(figsize=(8,5))
sns.barplot(data=df, x="day_type", y="cnt", estimator="mean", errorbar=None)
plt.title("Average Rentals: Working vs Non-working Days")
plt.xlabel("Day type")
plt.ylabel("Average rentals per hour")
plt.tight_layout()
plt.show()
display(df.groupby("day_type")["cnt"].agg(["count","mean","median","std"]).round(2))

### 6.4 Average demand by season

In [ ]:
season_labels = {1:"Spring", 2:"Summer", 3:"Fall", 4:"Winter"}
season_df = df.copy()
season_df["season_name"] = season_df["season"].map(season_labels)
season_summary = season_df.groupby("season_name", as_index=False)["cnt"].mean()
plt.figure(figsize=(8,5))
sns.barplot(data=season_summary, x="season_name", y="cnt",
            order=["Spring","Summer","Fall","Winter"], errorbar=None)
plt.title("Average Bike Rentals by Season")
plt.xlabel("Season")
plt.ylabel("Average rentals per hour")
plt.tight_layout()
plt.show()
display(season_summary.sort_values("cnt", ascending=False).round(2))

### 6.5 Weather and demand

In [ ]:
weather_labels = {1:"Clear / mostly clear", 2:"Mist / cloudy",
                  3:"Light rain or snow", 4:"Heavy rain or snow"}
weather_df = df.copy()
weather_df["weather_name"] = weather_df["weathersit"].map(weather_labels)
weather_summary = weather_df.groupby("weather_name", as_index=False)["cnt"].mean()
plt.figure(figsize=(10,5))
sns.barplot(data=weather_summary, x="weather_name", y="cnt", errorbar=None)
plt.title("Average Rentals by Weather Situation")
plt.xlabel("Weather")
plt.ylabel("Average rentals per hour")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()
display(weather_summary.sort_values("cnt", ascending=False).round(2))

### 6.6 Correlation heatmap

In [ ]:
corr_cols = ["temp","atemp","hum","windspeed","casual","registered","cnt"]
plt.figure(figsize=(9,6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### EDA observations — fill these after running the charts
1. Highest average demand hour: ______
2. Season with highest average demand: ______
3. Working-day versus non-working-day observation: ______
4. Weather-related observation: ______

Use the outputs as evidence. Do not assume a pattern before looking at the graphs.

## 7. Feature selection and preprocessing

**Target (`y`):** `cnt`.

**Predictors (`X`):** season, year, month, hour, holiday, weekday, working-day status, weather category, temperature, feels-like temperature, humidity and wind speed.

**Excluded:** `casual` and `registered` because they add up to `cnt`; `instant` because it is an identifier; `dteday` is used for chronological sorting but not as a raw feature.

Categorical variables are one-hot encoded. Numeric values are imputed and scaled. The preprocessing pipeline is fitted on training data only.

In [ ]:
categorical_features = ["season","yr","mnth","hr","holiday","weekday","workingday","weathersit"]
numeric_features = ["temp","atemp","hum","windspeed"]
features = categorical_features + numeric_features
target = "cnt"

X = df[features].copy()
y = df[target].astype(float).copy()

# Chronological holdout: first 80% train, last 20% test.
split = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Train period:", df["dteday"].iloc[0], "to", df["dteday"].iloc[split-1])
print("Test period:", df["dteday"].iloc[split], "to", df["dteday"].iloc[-1])
print("Feature columns:", features)

## 8. Build preprocessing and train models

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

estimators = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=20, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )
}
trained = {}
predictions = {}

for name, estimator in estimators.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    predictions[name] = np.maximum(0, pipe.predict(X_test))
    trained[name] = pipe
    print(name, "trained.")

## 9. Evaluate the models

- **MAE:** average absolute prediction error in rentals; lower is better.
- **RMSE:** penalizes large errors more than MAE; lower is better.
- **R²:** compares performance with a mean-baseline predictor; higher is generally better and can be negative.

In [ ]:
rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })
results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(results.round(3))

best_name = results.loc[0, "Model"]
best_model = trained[best_name]
best_pred = predictions[best_name]
print("Best model by lowest test RMSE:", best_name)

### 9.1 Compare model RMSE

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=results, x="Model", y="RMSE")
plt.title("Model Comparison by RMSE (Lower is Better)")
plt.ylabel("RMSE (rentals)")
plt.tight_layout()
plt.show()

### 9.2 Actual versus predicted

In [ ]:
compare = pd.DataFrame({"Actual rentals": y_test.to_numpy(), "Predicted rentals": best_pred})
plt.figure(figsize=(7,7))
sns.scatterplot(data=compare, x="Actual rentals", y="Predicted rentals", alpha=.45)
upper = max(compare.max().max(), 1)
plt.plot([0, upper], [0, upper], "r--", label="Ideal prediction")
plt.title(f"Actual vs Predicted — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()
display(compare.head(10).round(1))

## 10. Try one new prediction

The weather measurements below use the UCI dataset's normalized 0–1 scale, not degrees Celsius, percentage humidity, or km/h.

In [ ]:
sample = pd.DataFrame([{
    "season": 2, "yr": 1, "mnth": 6, "hr": 8,
    "holiday": 0, "weekday": 2, "workingday": 1, "weathersit": 1,
    "temp": 0.55, "atemp": 0.53, "hum": 0.60, "windspeed": 0.20
}])[features]

estimate = max(0, float(best_model.predict(sample)[0]))
display(sample)
print(f"Estimated rentals in this hour: {estimate:.0f}")
print("This is a model estimate, not a guarantee.")

## 11. Findings, conclusion and limitations

Fill this section with your actual outputs after running the notebook.

- Peak average demand hour: ______
- Highest average-demand season: ______
- Working-day pattern: ______
- Weather pattern: ______
- Best model by test RMSE: ______
- Best model MAE / RMSE / R²: ______ / ______ / ______

### Conclusion
This project followed a Data Science workflow to inspect, clean, visualize, and model hourly bike-sharing demand. EDA was used to explore patterns related to time and weather. Linear Regression and Random Forest were trained and evaluated on a held-out later portion of the dataset. The model with the lowest RMSE performed best on this particular split. The trained model can estimate hourly demand from selected conditions.

### Limitations
- This uses historical data, not live inventory or live weather.
- The weather variables are normalized.
- One chronological holdout is appropriate for a mini project but is not full time-series validation.
- Results depend on the dataset and model settings.

### References
1. UCI Bike Sharing Dataset: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset
2. Pandas: https://pandas.pydata.org/docs/
3. scikit-learn: https://scikit-learn.org/stable/
4. Seaborn: https://seaborn.pydata.org/
5. Matplotlib: https://matplotlib.org/stable/

**For your report:** save screenshots of the data-quality tables, EDA charts, model comparison and actual-versus-predicted graph. Use only results generated by your own run.